# load things for science :)

In [1]:
import pandas as pd


In [11]:

# ============================================================
# SETTINGS
# ============================================================

file = "/Users/aosmond/nuclearPhys/rga_info/CLAS12 RG-A Spring 2019.xlsx"       # Change this for each run period

# Currents you want to examine ----------------------- chooose background_currents depending on file above
# background_currents = [35, 45, 50] #sp18
# background_currents = [40, 45, 50, 55] #fa18
background_currents = [50] #sp19inb


# Number of simulated background events you want in total
total_sim_events = 10_000_000_000

# Read the spreadsheet
df = pd.read_excel(file, header=0)

# The spreadsheets have some columns whose useful names
# are actually on the next header row.
current_col = "Unnamed: 9"
events_col = "Total"
cook_col = "Cook?"
magnet_col = "Torus/Solenoid"

print("Using:")
print("  Production:", cook_col)
print("  Events:    ", events_col)
print("  Current:   ", current_col)
print("  Magnet:    ", magnet_col)


# ============================================================
# CLEAN PRODUCTION FLAG
# ============================================================

df["cook_clean"] = (
    df[cook_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ============================================================
# CLEAN EVENT COUNT
# ============================================================

# Total events in the spreadsheet are in millions.
df["events_M"] = pd.to_numeric(df[events_col], errors="coerce")


# ============================================================
# CLEAN CURRENT
# ============================================================

df["current_nA"] = pd.to_numeric(
    df[current_col],
    errors="coerce"
)

# ============================================================
# CLEAN MAGNET SETTING
# ============================================================

df["magnet_clean"] = (
    df[magnet_col]
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str.strip()
)

# Only keep the two standard RGA configurations
df["magnet_config"] = df["magnet_clean"].map({
    "-1/-1": "Inbending",
    "1/-1": "Outbending"
})

# ============================================================
# SELECT PRODUCTION RUNS
# ============================================================

production = df[
    df["cook_clean"].isin(["reg", "production"])
].copy()


# ============================================================
# KEEP ONLY INBENDING / OUTBENDING
# ============================================================

background = production[
    production["magnet_config"].isin(["Inbending", "Outbending"])
].copy()


# ============================================================
# SELECT THE BACKGROUND CURRENTS
# ============================================================

background = production[
    production["current_nA"].isin(background_currents)
].copy()

# background = production[
#     production["current_nA"].notna()
# ].copy()


# ============================================================
# PRINT BASIC INFORMATION
# ============================================================

print("\nProduction runs:")
print(len(production))

print("\nProduction runs at selected currents:")
print(len(background))


# ============================================================
# GROUP BY MAGNET + CURRENT
# ============================================================

summary = (
    background
    .groupby(
        ["magnet_config", "current_nA"],
        dropna=False
    )
    .agg(
        runs=("current_nA", "size"),
        events_M=("events_M", "sum")
    )
    .reset_index()
)


# ============================================================
# CONVERT EXPERIMENTAL EVENTS TO BILLIONS
# ============================================================

# summary["experimental_events_B"] = (
#     summary["events_M"] / 1000
# )

summary["exp_events_M"] = summary["events_M"]


# ============================================================
# CALCULATE EVENT FRACTIONS
# ============================================================

total_events_M = summary["events_M"].sum()


# ============================================================
# SIMULATION TARGET
# ============================================================

# Generate 10 billion events for EACH magnet configuration
sim_events_per_magnet = 10_000_000_000


# ============================================================
# CALCULATE EVENT FRACTIONS WITHIN EACH MAGNET CONFIGURATION
# ============================================================

summary["fraction"] = (
    summary["events_M"]
    / summary.groupby("magnet_config")["events_M"].transform("sum")
)

summary["percent"] = 100 * summary["fraction"]


# ============================================================
# CALCULATE SIMULATION EVENTS
# ============================================================

summary["sim_events"] = (
    summary["fraction"] * sim_events_per_magnet
)

# Convert simulated events to millions
summary["sim_events_M"] = (
    summary["sim_events"] / 1_000_000
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n==============================================")
print("BACKGROUND MIXTURE")
print("==============================================")

print(
    summary[
        [
            "magnet_config",
            "current_nA",
            "runs",
            "exp_events_M",
            "percent",
            "sim_events_M"
        ]
    ].to_string(
        index=False,
        formatters={
            "exp_events_M": "{:.3f}".format,
            "percent": "{:.3f}".format,
            "sim_events_M": "{:.3f}".format
        }
    )
)


# ============================================================
# TOTALS
# ============================================================

print("\n==============================================")
print("TOTALS")
print("==============================================")

for magnet in summary["magnet_config"].unique():

    magnet_data = summary[
        summary["magnet_config"] == magnet
    ]

    if len(magnet_data) == 0:
        continue

    exp_events_B = magnet_data["events_M"].sum() / 1000
    sim_events_B = magnet_data["sim_events_M"].sum() / 1000

    print(f"{magnet}:")
    print(f"    Experimental events: {exp_events_B:.3f} B")
    print(f"    Simulation events:   {sim_events_B:.3f} B")

print("\nOverall:")
print(f"    Experimental events: {summary['events_M'].sum() / 1000:.3f} B")
print(f"    Simulation events:   {summary['sim_events_M'].sum() / 1000:.3f} B")


# ============================================================
# SAVE RESULTS
# ============================================================

summary.to_csv(
    "background_mixture_summary.csv",
    index=False
)

Using:
  Production: Cook?
  Events:     Total
  Current:    Unnamed: 9
  Magnet:     Torus/Solenoid

Production runs:
137

Production runs at selected currents:
131

BACKGROUND MIXTURE
magnet_config  current_nA  runs exp_events_M percent sim_events_M
    Inbending        50.0   131    12138.277 100.000    10000.000

TOTALS
Inbending:
    Experimental events: 12.138 B
    Simulation events:   10.000 B

Overall:
    Experimental events: 12.138 B
    Simulation events:   10.000 B
